# Surper_GCG 12B Pipeline Notebook

This notebook is the single Colab entrypoint for the rebuilt 12B pipeline.

Use it only through `run_pipeline.py`.

Current runnable mainline:
- `t0_t1_bootstrap`
- `t0_t2_bootstrap`

Do not use old `L17/L23`-anchored wrappers as the 12B mainline.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone Or Update Repo

In [ ]:
%cd /content
import os

REPO_DIR = '/content/Surper_GCG'
REPO_URL = 'https://github.com/awaa-col/Surper_GCG_ResearchWorkFlow.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already exists: {REPO_DIR}')

%cd /content/Surper_GCG
!git pull --ff-only

## 3. Install Package

In [ ]:
%cd /content/Surper_GCG
!pip install -e .

## 4. Runtime Config

Edit only this cell before running the pipeline.

In [ ]:
import os

HF_TOKEN = ''
RUN_NAME = 'gemma3_12b_main'
MODEL = 'google/gemma-3-12b-it'
RESUME = True
N_TRAIN = 32
N_EVAL = 8
MAX_NEW_TOKENS = 96
SEED = 42

DRIVE_ROOT = '/content/drive/MyDrive/surper_gcg_pipeline_runs'
LOCAL_RESULTS_ROOT = '/content/Surper_GCG/results/pipeline_runs'
DRIVE_RUN_DIR = f'{DRIVE_ROOT}/{RUN_NAME}'
LOCAL_RUN_DIR = f'{LOCAL_RESULTS_ROOT}/{RUN_NAME}'

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

RESUME_FLAG = '--resume' if RESUME else ''

print({
    'RUN_NAME': RUN_NAME,
    'MODEL': MODEL,
    'RESUME': RESUME,
    'N_TRAIN': N_TRAIN,
    'N_EVAL': N_EVAL,
    'MAX_NEW_TOKENS': MAX_NEW_TOKENS,
    'SEED': SEED,
    'DRIVE_RUN_DIR': DRIVE_RUN_DIR,
    'LOCAL_RUN_DIR': LOCAL_RUN_DIR,
})

## 5. Inspect Available Pipeline Presets And Stages

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py --list-presets
!python run_pipeline.py --list-stages

## 6. Optional Restore From Drive

Use this after a Colab reset if you want to continue an existing run.

In [ ]:
import os
import shutil

os.makedirs(LOCAL_RESULTS_ROOT, exist_ok=True)

if not os.path.exists(DRIVE_RUN_DIR):
    print(f'No saved run found on Drive: {DRIVE_RUN_DIR}')
else:
    if os.path.exists(LOCAL_RUN_DIR):
        shutil.rmtree(LOCAL_RUN_DIR)
    shutil.copytree(DRIVE_RUN_DIR, LOCAL_RUN_DIR)
    print(f'Restored run: {DRIVE_RUN_DIR} -> {LOCAL_RUN_DIR}')

## 7. Dry Run `T0-T1`

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py \
  --preset t0_t1_bootstrap \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --seed "$SEED" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## 8. Run `T0-T1`

Use this as the first real mainline run.

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py \
  --preset t0_t1_bootstrap \
  --run-name "$RUN_NAME" \
  $RESUME_FLAG \
  --model "$MODEL" \
  --seed "$SEED" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## 9. Check `T0-T1` Artifacts

Review these before advancing to gate discovery.

In [ ]:
!ls -lah "$LOCAL_RUN_DIR"
!test -f "$LOCAL_RUN_DIR/exp11_review_pack.jsonl" && echo 'Found exp11_review_pack.jsonl'
!test -f "$LOCAL_RUN_DIR/exp00_diagnosis.json" && echo 'Found exp00_diagnosis.json'
!test -f "$LOCAL_RUN_DIR/pipeline_manifest.json" && echo 'Found pipeline_manifest.json'
!test -f "$LOCAL_RUN_DIR/pipeline_stage_summary.md" && echo 'Found pipeline_stage_summary.md'

## 10. Dry Run `T0-T2`

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py \
  --preset t0_t2_bootstrap \
  --run-name "$RUN_NAME" \
  --model "$MODEL" \
  --seed "$SEED" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS" \
  --dry-run

## 11. Run `T0-T2`

Run this only after reviewing `T0-T1`.

In [ ]:
%cd /content/Surper_GCG
!python run_pipeline.py \
  --preset t0_t2_bootstrap \
  --run-name "$RUN_NAME" \
  $RESUME_FLAG \
  --model "$MODEL" \
  --seed "$SEED" \
  --n-train "$N_TRAIN" \
  --n-eval "$N_EVAL" \
  --max-new-tokens "$MAX_NEW_TOKENS"

## 12. Inspect Final Artifacts

In [ ]:
!ls -lah /content/Surper_GCG/results/pipeline_runs
!ls -lah "$LOCAL_RUN_DIR"

## 13. Sync Run Back To Drive

In [ ]:
import os
import shutil

os.makedirs(DRIVE_ROOT, exist_ok=True)
if os.path.exists(DRIVE_RUN_DIR):
    shutil.rmtree(DRIVE_RUN_DIR)
shutil.copytree(LOCAL_RUN_DIR, DRIVE_RUN_DIR)
print(f'Saved run to Drive: {DRIVE_RUN_DIR}')

## 14. Optional Zip Export

In [ ]:
!zip -r "/content/drive/MyDrive/${RUN_NAME}_pipeline_results.zip" "$LOCAL_RUN_DIR"